In [2]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lit
)

# Fresh read of authoritative BTS source tables
t100 = spark.table("bronze_t100_domestic_segment")

print("T-100 Rows:", t100.count())

print("\nT-100 Schema")
t100.printSchema()

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 4, Finished, Available, Finished, False)

T-100 Rows: 36186

T-100 Schema
root
 |-- departures_scheduled: long (nullable = true)
 |-- payload: long (nullable = true)
 |-- seats: long (nullable = true)
 |-- PASSENGERS: long (nullable = true)
 |-- FREIGHT: long (nullable = true)
 |-- MAIL: long (nullable = true)
 |-- DISTANCE: long (nullable = true)
 |-- RAMP_TO_RAMP: long (nullable = true)
 |-- AIR_TIME: long (nullable = true)
 |-- UNIQUE_CARRIER: string (nullable = true)
 |-- AIRLINE_ID: long (nullable = true)
 |-- UNIQUE_CARRIER_NAME: string (nullable = true)
 |-- CARRIER: string (nullable = true)
 |-- CARRIER_NAME: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: long (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_AIRPORT_ID: long (nullable = true)
 |-- DEST: string (nullable = true)
 |-- AIRCRAFT_TYPE: long (nullable = true)
 |-- YEAR: long (nullable = true)
 |-- QUARTER: long (nullable = true)
 |-- MONTH: long (nullable = true)



In [3]:
t100_airline_columns = [
    c for c in t100.columns
    if "AIRLINE" in c.upper() or "CARRIER" in c.upper()
]

t100_airline_columns

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 5, Finished, Available, Finished, False)

['UNIQUE_CARRIER',
 'AIRLINE_ID',
 'UNIQUE_CARRIER_NAME',
 'CARRIER',
 'CARRIER_NAME']

In [4]:
t100.select(
    *[col(c) for c in t100_airline_columns]
).distinct().show(200, truncate=False)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 6, Finished, Available, Finished, False)

+--------------+----------+-----------------------------------------------------------------------------+-------+-----------------------------------------------------------------------------+
|UNIQUE_CARRIER|AIRLINE_ID|UNIQUE_CARRIER_NAME                                                          |CARRIER|CARRIER_NAME                                                                 |
+--------------+----------+-----------------------------------------------------------------------------+-------+-----------------------------------------------------------------------------+
|WRD           |20285     |Ward Air                                                                     |WRD    |Ward Air                                                                     |
|QX            |19687     |Horizon Air                                                                  |QX     |Horizon Air                                                                  |
|C5            |20445     |CommuteAir LL

In [5]:
t100_id_quality = (
    t100
    .filter(col("AIRLINE_ID").isNotNull())
    .groupBy("AIRLINE_ID")
    .agg(
        F.countDistinct("UNIQUE_CARRIER").alias("unique_carrier_count"),
        F.countDistinct("UNIQUE_CARRIER_NAME").alias("carrier_name_count")
    )
    .filter(
        (col("unique_carrier_count") > 1) |
        (col("carrier_name_count") > 1)
    )
)

print("T-100 Airline ID Conflicts:", t100_id_quality.count())

display(t100_id_quality)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 7, Finished, Available, Finished, False)

T-100 Airline ID Conflicts: 0


SynapseWidget(Synapse.DataFrame, 2e062204-e6cb-4c21-9312-feca1cc91565)

In [6]:
t100_code_quality = (
    t100
    .filter(col("UNIQUE_CARRIER").isNotNull())
    .groupBy("UNIQUE_CARRIER")
    .agg(
        F.countDistinct("AIRLINE_ID").alias("airline_id_count")
    )
    .filter(col("airline_id_count") > 1)
)

print(
    "T-100 Unique Carrier Conflicts:",
    t100_code_quality.count()
)

display(t100_code_quality)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 8, Finished, Available, Finished, False)

T-100 Unique Carrier Conflicts: 0


SynapseWidget(Synapse.DataFrame, 57eee3dc-d11b-41bf-b87c-71ade0273a43)

In [7]:
fuel_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "Files/raw/batch/form41/"
        "fuel_cost_consumption_2024_jan.csv"
    )
)

print("Form 41 Rows:", fuel_raw.count())

fuel_raw.printSchema()

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 9, Finished, Available, Finished, False)

Form 41 Rows: 53
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- AIRLINE_ID: integer (nullable = true)
 |-- UNIQUE_CARRIER: string (nullable = true)
 |-- CARRIER_NAME: string (nullable = true)
 |-- TDOMT_GALLONS: double (nullable = true)
 |-- TINT_GALLONS: double (nullable = true)
 |-- TOTAL_GALLONS: double (nullable = true)
 |-- TDOMT_COST: double (nullable = true)
 |-- TINT_COST: double (nullable = true)
 |-- TOTAL_COST: double (nullable = true)



In [8]:
fuel_airline_columns = [
    c for c in fuel_raw.columns
    if "AIRLINE" in c.upper()
    or "CARRIER" in c.upper()
]

fuel_airline_columns

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 10, Finished, Available, Finished, False)

['AIRLINE_ID', 'UNIQUE_CARRIER', 'CARRIER_NAME']

In [10]:
fuel_raw.select(
    *[col(c) for c in fuel_airline_columns]
).distinct().orderBy(
    "AIRLINE_ID"
).show(100, truncate=False)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 12, Finished, Available, Finished, False)

+----------+--------------+------------------------------------------------------------------------+
|AIRLINE_ID|UNIQUE_CARRIER|CARRIER_NAME                                                            |
+----------+--------------+------------------------------------------------------------------------+
|19393     |WN            |Southwest Airlines Co.                                                  |
|19687     |QX            |Horizon Air                                                             |
|19690     |HA            |Hawaiian Airlines Inc.                                                  |
|19790     |DL            |Delta Air Lines Inc.                                                    |
|19805     |AA            |American Airlines Inc.                                                  |
|19874     |8C            |Air Transport International                                             |
|19917     |5X            |United Parcel Service                                           

In [13]:
fuel_id_quality = (
    fuel_raw
    .filter(col("AIRLINE_ID").isNotNull())
    .groupBy("AIRLINE_ID")
    .agg(
        F.countDistinct("UNIQUE_CARRIER")
        .alias("unique_carrier_count"),

        F.countDistinct("CARRIER_NAME")
        .alias("carrier_name_count")
    )
    .filter(
        (col("unique_carrier_count") > 1) |
        (col("carrier_name_count") > 1)
    )
)

fuel_code_quality = (
    fuel_raw
    .filter(col("UNIQUE_CARRIER").isNotNull())
    .groupBy("UNIQUE_CARRIER")
    .agg(
        F.countDistinct("AIRLINE_ID")
        .alias("airline_id_count")
    )
    .filter(
        col("airline_id_count") > 1
    )
)

print(
    "Form 41 Airline ID Conflicts:",
    fuel_id_quality.count()
)

print(
    "Form 41 Unique Carrier Conflicts:",
    fuel_code_quality.count()
)

display(fuel_id_quality)
display(fuel_code_quality)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 15, Finished, Available, Finished, False)

Form 41 Airline ID Conflicts: 0
Form 41 Unique Carrier Conflicts: 0


SynapseWidget(Synapse.DataFrame, bfd2ee33-204c-497e-a330-0be519045da4)

SynapseWidget(Synapse.DataFrame, a98e0e05-b49a-423c-ad9f-2b65356a8c55)

In [14]:
t100_carriers = (
    t100
    .select(
        col("AIRLINE_ID").cast("long").alias("airline_id"),
        upper(trim(col("UNIQUE_CARRIER"))).alias("unique_carrier"),
        trim(col("UNIQUE_CARRIER_NAME")).alias("carrier_name")
    )
    .filter(
        col("airline_id").isNotNull() &
        col("unique_carrier").isNotNull()
    )
    .dropDuplicates()
)

fuel_carriers = (
    fuel_raw
    .select(
        col("AIRLINE_ID").cast("long").alias("airline_id"),
        upper(trim(col("UNIQUE_CARRIER"))).alias("unique_carrier"),
        trim(col("CARRIER_NAME")).alias("carrier_name")
    )
    .filter(
        col("airline_id").isNotNull() &
        col("unique_carrier").isNotNull()
    )
    .dropDuplicates()
)

print("Distinct T-100 Carriers:", t100_carriers.count())
print("Distinct Form 41 Carriers:", fuel_carriers.count())

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 16, Finished, Available, Finished, False)

Distinct T-100 Carriers: 153
Distinct Form 41 Carriers: 53


In [15]:
cross_source_conflicts = (
    t100_carriers.alias("t")
    .join(
        fuel_carriers.alias("f"),
        col("t.airline_id") == col("f.airline_id"),
        "inner"
    )
    .filter(
        col("t.unique_carrier") != col("f.unique_carrier")
    )
    .select(
        col("t.airline_id"),
        col("t.unique_carrier").alias("t100_carrier"),
        col("f.unique_carrier").alias("form41_carrier"),
        col("t.carrier_name").alias("t100_name"),
        col("f.carrier_name").alias("form41_name")
    )
)

print(
    "Cross-Source Airline ID Conflicts:",
    cross_source_conflicts.count()
)

display(cross_source_conflicts)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 17, Finished, Available, Finished, False)

Cross-Source Airline ID Conflicts: 0


SynapseWidget(Synapse.DataFrame, f9d4102f-0e7e-48c2-a168-65b56e5fd71e)

In [16]:
form41_not_in_t100 = (
    fuel_carriers.alias("f")
    .join(
        t100_carriers.alias("t"),
        col("f.airline_id") == col("t.airline_id"),
        "left_anti"
    )
)

print(
    "Form 41 Carriers Not In T-100:",
    form41_not_in_t100.count()
)

display(form41_not_in_t100)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 18, Finished, Available, Finished, False)

Form 41 Carriers Not In T-100: 2


SynapseWidget(Synapse.DataFrame, 39860350-3a3f-4de3-a759-ca81794968de)

In [17]:
all_bts_carriers = (
    t100_carriers
    .unionByName(fuel_carriers)
)

candidate_dim_airline = (
    all_bts_carriers
    .groupBy(
        "airline_id",
        "unique_carrier"
    )
    .agg(
        F.first(
            "carrier_name",
            ignorenulls=True
        ).alias("airline_name")
    )
)

print(
    "Candidate Airline Rows:",
    candidate_dim_airline.count()
)

candidate_dim_airline.orderBy(
    "airline_id"
).show(200, truncate=False)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 19, Finished, Available, Finished, False)

Candidate Airline Rows: 155
+----------+--------------+-----------------------------------------------------------------------------+
|airline_id|unique_carrier|airline_name                                                                 |
+----------+--------------+-----------------------------------------------------------------------------+
|19393     |WN            |Southwest Airlines Co.                                                       |
|19531     |AC            |Air Canada                                                                   |
|19534     |AM            |Aeromexico                                                                   |
|19543     |CA            |Air China                                                                    |
|19544     |CI            |China Airlines Ltd.                                                          |
|19548     |JL            |Japan Air Lines Co. Ltd.                                                     |
|19550     |KE    

In [18]:
from pyspark.sql.window import Window

airline_key_window = Window.orderBy(
    "airline_id"
)

candidate_dim_airline = (
    candidate_dim_airline
    .withColumn(
        "airline_key",
        F.row_number().over(
            airline_key_window
        )
    )
    .select(
        "airline_key",
        "airline_id",
        "unique_carrier",
        "airline_name"
    )
)

candidate_dim_airline.printSchema()

candidate_dim_airline.orderBy(
    "airline_key"
).show(200, truncate=False)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 20, Finished, Available, Finished, False)

root
 |-- airline_key: integer (nullable = false)
 |-- airline_id: long (nullable = true)
 |-- unique_carrier: string (nullable = true)
 |-- airline_name: string (nullable = true)

+-----------+----------+--------------+-----------------------------------------------------------------------------+
|airline_key|airline_id|unique_carrier|airline_name                                                                 |
+-----------+----------+--------------+-----------------------------------------------------------------------------+
|1          |19393     |WN            |Southwest Airlines Co.                                                       |
|2          |19531     |AC            |Air Canada                                                                   |
|3          |19534     |AM            |Aeromexico                                                                   |
|4          |19543     |CA            |Air China                                                               

In [19]:
print(
    "Candidate Rows:",
    candidate_dim_airline.count()
)

print(
    "Distinct Airline IDs:",
    candidate_dim_airline
    .select("airline_id")
    .distinct()
    .count()
)

print(
    "Distinct Unique Carriers:",
    candidate_dim_airline
    .select("unique_carrier")
    .distinct()
    .count()
)

print(
    "Null Airline Keys:",
    candidate_dim_airline
    .filter(col("airline_key").isNull())
    .count()
)

print(
    "Null Airline IDs:",
    candidate_dim_airline
    .filter(col("airline_id").isNull())
    .count()
)

print(
    "Null Unique Carriers:",
    candidate_dim_airline
    .filter(col("unique_carrier").isNull())
    .count()
)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 21, Finished, Available, Finished, False)

Candidate Rows: 155
Distinct Airline IDs: 155
Distinct Unique Carriers: 155
Null Airline Keys: 0
Null Airline IDs: 0
Null Unique Carriers: 0


In [20]:
# Read current Silver flight dataset
silver_flights = spark.table("silver_flights")

flight_airline_coverage = (
    silver_flights.alias("f")
    .join(
        candidate_dim_airline.alias("a"),
        upper(trim(col("f.OP_UNIQUE_CARRIER")))
        == col("a.unique_carrier"),
        "left"
    )
)

print(
    "Silver Flight Rows:",
    silver_flights.count()
)

print(
    "Rows After Airline Join:",
    flight_airline_coverage.count()
)

print(
    "Null Airline Keys:",
    flight_airline_coverage
    .filter(col("a.airline_key").isNull())
    .count()
)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 22, Finished, Available, Finished, False)

Silver Flight Rows: 547271
Rows After Airline Join: 547271
Null Airline Keys: 0


In [21]:
unmatched_flight_carriers = (
    flight_airline_coverage
    .filter(col("a.airline_key").isNull())
    .select(
        col("f.OP_UNIQUE_CARRIER")
    )
    .distinct()
)

print(
    "Unmatched Flight Carrier Codes:",
    unmatched_flight_carriers.count()
)

display(unmatched_flight_carriers)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 23, Finished, Available, Finished, False)

Unmatched Flight Carrier Codes: 0


SynapseWidget(Synapse.DataFrame, dc6ab4cf-1540-4120-9d70-4e07ed8d1379)

In [22]:
candidate_dim_airline.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_airline")

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 24, Finished, Available, Finished, False)

In [23]:
dim_airline_check = spark.table("dim_airline")

print("dim_airline Rows:", dim_airline_check.count())

dim_airline_check.printSchema()

dim_airline_check.orderBy(
    "airline_key"
).show(20, truncate=False)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 25, Finished, Available, Finished, False)

dim_airline Rows: 155
root
 |-- airline_key: integer (nullable = true)
 |-- airline_id: long (nullable = true)
 |-- unique_carrier: string (nullable = true)
 |-- airline_name: string (nullable = true)

+-----------+----------+--------------+----------------------------------------+
|airline_key|airline_id|unique_carrier|airline_name                            |
+-----------+----------+--------------+----------------------------------------+
|1          |19393     |WN            |Southwest Airlines Co.                  |
|2          |19531     |AC            |Air Canada                              |
|3          |19534     |AM            |Aeromexico                              |
|4          |19543     |CA            |Air China                               |
|5          |19544     |CI            |China Airlines Ltd.                     |
|6          |19548     |JL            |Japan Air Lines Co. Ltd.                |
|7          |19550     |KE            |Korean Air Lines Co. Ltd.     

In [24]:
print(
    "Distinct Airline Keys:",
    dim_airline_check
    .select("airline_key")
    .distinct()
    .count()
)

print(
    "Distinct Airline IDs:",
    dim_airline_check
    .select("airline_id")
    .distinct()
    .count()
)

print(
    "Distinct Unique Carriers:",
    dim_airline_check
    .select("unique_carrier")
    .distinct()
    .count()
)

StatementMeta(, 6ac6f55b-dbbd-4f86-b318-190804b60343, 26, Finished, Available, Finished, False)

Distinct Airline Keys: 155
Distinct Airline IDs: 155
Distinct Unique Carriers: 155
